# Cell 1 — Forecasting Pipeline Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = PROJECT_ROOT / "models" / "random_forest_demand_model.pkl"
FEATURE_DIR = PROJECT_ROOT / "models" / "features"
DATA_DIR = PROJECT_ROOT / "Data" / "raw"

X_train = pd.read_csv(
    FEATURE_DIR / "X_train.csv"
)

X_test = pd.read_csv(
    FEATURE_DIR / "X_test.csv"
)

y_train = pd.read_csv(
    FEATURE_DIR / "y_train.csv"
).squeeze("columns")

y_test = pd.read_csv(
    FEATURE_DIR / "y_test.csv"
).squeeze("columns")

feature_columns = pd.read_csv(
    FEATURE_DIR / "feature_columns.csv"
)["Feature"].tolist()

sales_df = pd.read_csv(
    DATA_DIR / "sales.csv"
)

product_df = pd.read_csv(
    DATA_DIR / "product_master.csv"
)

inventory_df = pd.read_csv(
    DATA_DIR / "inventory.csv"
)

sales_df["Date"] = pd.to_datetime(
    sales_df["Date"]
)

inventory_df["Snapshot_Date"] = pd.to_datetime(
    inventory_df["Snapshot_Date"]
)

model = joblib.load(
    MODEL_PATH
)

X_test = X_test[feature_columns]

print("=" * 80)
print("FORESIGHT — DEMAND FORECASTING PIPELINE")
print("=" * 80)

print("\nModel:")
print(MODEL_PATH)

print("\nX_test Shape:")
print(X_test.shape)

print("\nSales Shape:")
print(sales_df.shape)

print("\nProduct Master Shape:")
print(product_df.shape)

print("\nInventory Shape:")
print(inventory_df.shape)

print("\nFeatures:")
print(len(feature_columns))

print("\nModel Loaded:")
print(type(model).__name__)

print("\n✅ Forecasting pipeline setup completed successfully.")

FORESIGHT — DEMAND FORECASTING PIPELINE

Model:
d:\Zidio project\FORESIGHT Project\models\random_forest_demand_model.pkl

X_test Shape:
(7240, 130)

Sales Shape:
(36550, 6)

Product Master Shape:
(50, 8)

Inventory Shape:
(4800, 8)

Features:
130

Model Loaded:
RandomForestRegressor

✅ Forecasting pipeline setup completed successfully.


# Cell 2 — Demand Prediction.

In [2]:
forecast_prediction = model.predict(X_test)

forecast_df = pd.DataFrame({
    "Predicted_Demand": forecast_prediction
})

forecast_df["Predicted_Demand"] = forecast_df[
    "Predicted_Demand"
].clip(lower=0)

print("=" * 80)
print("DEMAND FORECAST GENERATION")
print("=" * 80)

print("\nForecast Rows:")
print(len(forecast_df))

print("\nPrediction Statistics:")

print(
    "Minimum:",
    round(forecast_df["Predicted_Demand"].min(), 2)
)

print(
    "Maximum:",
    round(forecast_df["Predicted_Demand"].max(), 2)
)

print(
    "Mean:",
    round(forecast_df["Predicted_Demand"].mean(), 2)
)

print(
    "Median:",
    round(forecast_df["Predicted_Demand"].median(), 2)
)

print("\nSample Forecasts:")
display(forecast_df.head(10))

print("\n✅ Demand forecast generated successfully.")

DEMAND FORECAST GENERATION

Forecast Rows:
7240

Prediction Statistics:
Minimum: 2.66
Maximum: 42.57
Mean: 15.69
Median: 13.89

Sample Forecasts:


,Predicted_Demand
0,5.130627
1,11.960100
2,11.630500
3,11.625223
4,11.272866
5,12.423235
6,11.602157
7,10.458079
8,11.131969
9,10.179554



✅ Demand forecast generated successfully.


# Cell 3 — Map Forecasts to Date & SKU

SKU-LEVEL DEMAND FORECAST

Forecast Rows:
7240

Date Range:
2025-08-09 00:00:00 to 2025-12-31 00:00:00

Unique SKUs:
50

Sample Forecasts:


,Date,SKU,Predicted_Demand
0,2025-08-09,SKU011,5.13
1,2025-08-09,SKU012,11.96
2,2025-08-09,SKU013,11.63
3,2025-08-09,SKU014,11.63
4,2025-08-09,SKU015,11.27
5,2025-08-09,SKU016,12.42
6,2025-08-09,SKU017,11.60
7,2025-08-09,SKU018,10.46
8,2025-08-09,SKU019,11.13
9,2025-08-09,SKU020,10.18



Forecast Summary by SKU:


,SKU,Forecast_Mean,Forecast_Total,Forecast_Max
0,SKU001,15.55,2239.85,35.86
1,SKU002,15.61,2248.51,36.22
2,SKU003,15.73,2265.73,39.95
3,SKU004,15.75,2267.31,39.82
4,SKU005,15.86,2284.16,34.21
5,SKU006,15.75,2268.09,36.00
6,SKU007,15.76,2269.84,35.92
7,SKU008,15.72,2263.76,34.79
8,SKU009,15.77,2271.03,36.68
9,SKU010,15.82,2278.46,36.72



✅ SKU-level demand forecast created successfully.


# Cell 4 — Integrate Product & Inventory Data

In [ ]:
product_features = product_df[
    [
        "SKU",
        "Product_Name",
        "Category",
        "Subcategory",
        "Cost_Price",
        "Selling_Price",
        "Gross_Margin_Per_Unit"
    ]
].drop_duplicates("SKU")

latest_inventory = (
    inventory_df
    .sort_values(["SKU", "Snapshot_Date"])
    .groupby("SKU")
    .tail(1)
)

inventory_features = latest_inventory[
    [
        "SKU",
        "Snapshot_Date",
        "Current_Stock",
        "On_Order",
        "Lead_Time_Days",
        "Safety_Stock",
        "Reorder_Point",
        "Inventory_Value"
    ]
].copy()

forecast_output = forecast_output.merge(
    product_features,
    on="SKU",
    how="left"
)

forecast_output = forecast_output.merge(
    inventory_features,
    on="SKU",
    how="left"
)

print("=" * 80)
print("FORECAST + PRODUCT + INVENTORY INTEGRATION")
print("=" * 80)

print("\nForecast Shape:")
print(forecast_output.shape)

print("\nUnique SKUs:")
print(forecast_output["SKU"].nunique())

print("\nMissing Values:")
display(
    forecast_output[
        [
            "Product_Name",
            "Category",
            "Current_Stock",
            "On_Order",
            "Lead_Time_Days",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ].isna().sum().to_frame("Missing_Count")
)

print("\nIntegrated Forecast Sample:")

display(
    forecast_output[
        [
            "Date",
            "SKU",
            "Product_Name",
            "Category",
            "Predicted_Demand",
            "Current_Stock",
            "On_Order",
            "Lead_Time_Days",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ].head(10)
)

print("\n✅ Product and inventory data integrated successfully.")

FORECAST + PRODUCT + INVENTORY INTEGRATION

Forecast Shape:
(7240, 16)

Unique SKUs:
50

Missing Values:


,Missing_Count
Product_Name,0
Category,0
Current_Stock,0
On_Order,0
Lead_Time_Days,0
Safety_Stock,0
Reorder_Point,0



Integrated Forecast Sample:


,Date,SKU,Product_Name,Category,Predicted_Demand,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point
0,2025-08-09,SKU011,Product 011,Furniture,5.13,753,269,3,179,247
1,2025-08-09,SKU012,Product 012,Home Decor,11.96,47,60,12,16,58
2,2025-08-09,SKU013,Product 013,Kitchen,11.63,151,95,4,32,61
3,2025-08-09,SKU014,Product 014,Lighting,11.63,309,334,10,130,299
4,2025-08-09,SKU015,Product 015,Storage,11.27,187,245,9,119,265
5,2025-08-09,SKU016,Product 016,Furniture,12.42,255,187,11,30,129
6,2025-08-09,SKU017,Product 017,Home Decor,11.60,95,26,12,108,274
7,2025-08-09,SKU018,Product 018,Kitchen,10.46,235,102,8,66,184
8,2025-08-09,SKU019,Product 019,Lighting,11.13,227,114,14,25,126
9,2025-08-09,SKU020,Product 020,Storage,10.18,1002,353,3,129,227



✅ Product and inventory data integrated successfully.


# Cell 5 — Inventory Coverage & Stockout Risk

In [ ]:
forecast_output["Lead_Time_Demand"] = (
    forecast_output["Predicted_Demand"] *
    forecast_output["Lead_Time_Days"]
)

forecast_output["Available_Inventory"] = (
    forecast_output["Current_Stock"] +
    forecast_output["On_Order"]
)

forecast_output["Days_of_Inventory"] = np.where(
    forecast_output["Predicted_Demand"] > 0,
    forecast_output["Current_Stock"] /
    forecast_output["Predicted_Demand"],
    np.inf
)

forecast_output["Stock_Coverage_After_Order"] = np.where(
    forecast_output["Predicted_Demand"] > 0,
    forecast_output["Available_Inventory"] /
    forecast_output["Predicted_Demand"],
    np.inf
)

forecast_output["Stockout_Gap"] = (
    forecast_output["Lead_Time_Demand"] -
    forecast_output["Current_Stock"]
)

forecast_output["Stockout_Risk"] = np.where(
    forecast_output["Current_Stock"] < forecast_output["Lead_Time_Demand"],
    "High",
    np.where(
        forecast_output["Current_Stock"] <
        forecast_output["Lead_Time_Demand"] +
        forecast_output["Safety_Stock"],
        "Medium",
        "Low"
    )
)

forecast_output["Stock_Status"] = np.where(
    forecast_output["Current_Stock"] <=
    forecast_output["Reorder_Point"],
    "Reorder Required",
    np.where(
        forecast_output["Current_Stock"] <
        forecast_output["Safety_Stock"],
        "Below Safety Stock",
        "Sufficient Stock"
    )
)

print("=" * 80)
print("INVENTORY COVERAGE & STOCKOUT RISK")
print("=" * 80)

print("\nStockout Risk Distribution:")

display(
    forecast_output["Stockout_Risk"]
    .value_counts()
    .rename_axis("Risk_Level")
    .reset_index(name="SKU_Day_Count")
)

print("\nStock Status Distribution:")

display(
    forecast_output["Stock_Status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="SKU_Day_Count")
)

print("\nInventory Risk Sample:")

display(
    forecast_output[
        [
            "Date",
            "SKU",
            "Predicted_Demand",
            "Current_Stock",
            "On_Order",
            "Lead_Time_Days",
            "Lead_Time_Demand",
            "Days_of_Inventory",
            "Stockout_Gap",
            "Stockout_Risk",
            "Stock_Status"
        ]
    ].head(15)
)

print("\nHigh Stockout Risk:",
      (forecast_output["Stockout_Risk"] == "High").sum())

print("Medium Stockout Risk:",
      (forecast_output["Stockout_Risk"] == "Medium").sum())

print("Low Stockout Risk:",
      (forecast_output["Stockout_Risk"] == "Low").sum())

print("\n✅ Inventory coverage and stockout risk analysis completed successfully.")

INVENTORY COVERAGE & STOCKOUT RISK

Stockout Risk Distribution:


,Risk_Level,SKU_Day_Count
0,Low,4296
1,High,1768
2,Medium,1176



Stock Status Distribution:


,Status,SKU_Day_Count
0,Sufficient Stock,5067
1,Reorder Required,2173



Inventory Risk Sample:


,Date,SKU,Predicted_Demand,Current_Stock,On_Order,Lead_Time_Days,Lead_Time_Demand,Days_of_Inventory,Stockout_Gap,Stockout_Risk,Stock_Status
0,2025-08-09,SKU011,5.13,753,269,3,15.39,146.783626,-737.61,Low,Sufficient Stock
1,2025-08-09,SKU012,11.96,47,60,12,143.52,3.929766,96.52,High,Reorder Required
2,2025-08-09,SKU013,11.63,151,95,4,46.52,12.983663,-104.48,Low,Sufficient Stock
3,2025-08-09,SKU014,11.63,309,334,10,116.30,26.569218,-192.70,Low,Sufficient Stock
4,2025-08-09,SKU015,11.27,187,245,9,101.43,16.592724,-85.57,Medium,Reorder Required
5,2025-08-09,SKU016,12.42,255,187,11,136.62,20.531401,-118.38,Low,Sufficient Stock
6,2025-08-09,SKU017,11.60,95,26,12,139.20,8.189655,44.20,High,Reorder Required
7,2025-08-09,SKU018,10.46,235,102,8,83.68,22.466539,-151.32,Low,Sufficient Stock
8,2025-08-09,SKU019,11.13,227,114,14,155.82,20.395328,-71.18,Low,Sufficient Stock
9,2025-08-09,SKU020,10.18,1002,353,3,30.54,98.428291,-971.46,Low,Sufficient Stock



High Stockout Risk: 1768
Medium Stockout Risk: 1176
Low Stockout Risk: 4296

✅ Inventory coverage and stockout risk analysis completed successfully.


# Cell 6 — Overstock Risk Analysis

In [6]:
forecast_output["Expected_Demand_During_Coverage"] = (
    forecast_output["Predicted_Demand"] *
    forecast_output["Lead_Time_Days"]
)

forecast_output["Excess_Inventory"] = (
    forecast_output["Current_Stock"] -
    forecast_output["Expected_Demand_During_Coverage"] -
    forecast_output["Safety_Stock"]
)

forecast_output["Excess_Inventory"] = (
    forecast_output["Excess_Inventory"].clip(lower=0)
)

forecast_output["Overstock_Risk"] = np.where(
    forecast_output["Excess_Inventory"] >
    forecast_output["Predicted_Demand"] * 14,
    "High",
    np.where(
        forecast_output["Excess_Inventory"] >
        forecast_output["Predicted_Demand"] * 7,
        "Medium",
        "Low"
    )
)

forecast_output["Inventory_Health"] = np.select(
    [
        forecast_output["Stockout_Risk"] == "High",
        forecast_output["Overstock_Risk"] == "High",
        forecast_output["Stockout_Risk"] == "Medium",
        forecast_output["Overstock_Risk"] == "Medium"
    ],
    [
        "Stockout Risk",
        "Overstock Risk",
        "Watch - Low Stock",
        "Watch - Excess Stock"
    ],
    default="Healthy"
)

print("=" * 80)
print("OVERSTOCK RISK ANALYSIS")
print("=" * 80)

print("\nOverstock Risk Distribution:")

overstock_summary = (
    forecast_output["Overstock_Risk"]
    .value_counts()
    .rename_axis("Risk_Level")
    .reset_index(name="SKU_Day_Count")
)

display(overstock_summary)

print("\nInventory Health Distribution:")

health_summary = (
    forecast_output["Inventory_Health"]
    .value_counts()
    .rename_axis("Inventory_Health")
    .reset_index(name="SKU_Day_Count")
)

display(health_summary)

print("\nExcess Inventory Statistics:")

print(
    "Total Excess Inventory:",
    round(
        forecast_output["Excess_Inventory"].sum(),
        2
    )
)

print(
    "Average Excess Inventory:",
    round(
        forecast_output["Excess_Inventory"].mean(),
        2
    )
)

print(
    "Maximum Excess Inventory:",
    round(
        forecast_output["Excess_Inventory"].max(),
        2
    )
)

print("\nTop 10 Overstock Risks:")

top_overstock = (
    forecast_output[
        [
            "SKU",
            "Product_Name",
            "Category",
            "Predicted_Demand",
            "Current_Stock",
            "Safety_Stock",
            "Excess_Inventory",
            "Overstock_Risk"
        ]
    ]
    .sort_values(
        "Excess_Inventory",
        ascending=False
    )
    .head(10)
)

display(top_overstock)

print("\n✅ Overstock risk analysis completed successfully.")

OVERSTOCK RISK ANALYSIS

Overstock Risk Distribution:


,Risk_Level,SKU_Day_Count
0,Low,4313
1,High,1994
2,Medium,933



Inventory Health Distribution:


,Inventory_Health,SKU_Day_Count
0,Overstock Risk,1994
1,Stockout Risk,1768
2,Healthy,1369
3,Watch - Low Stock,1176
4,Watch - Excess Stock,933



Excess Inventory Statistics:
Total Excess Inventory: 1006639.06
Average Excess Inventory: 139.04
Maximum Excess Inventory: 930.9

Top 10 Overstock Risks:


,SKU,Product_Name,Category,Predicted_Demand,Current_Stock,Safety_Stock,Excess_Inventory,Overstock_Risk
7114,SKU025,Product 025,Storage,5.15,1124,121,930.90,High
6764,SKU025,Product 025,Storage,5.17,1124,121,930.62,High
7164,SKU025,Product 025,Storage,5.39,1124,121,927.54,High
6664,SKU025,Product 025,Storage,5.55,1124,121,925.30,High
6814,SKU025,Product 025,Storage,5.57,1124,121,925.02,High
7014,SKU025,Product 025,Storage,5.92,1124,121,920.12,High
6714,SKU025,Product 025,Storage,6.04,1124,121,918.44,High
7214,SKU025,Product 025,Storage,6.09,1124,121,917.74,High
6864,SKU025,Product 025,Storage,6.16,1124,121,916.76,High
3914,SKU025,Product 025,Storage,6.25,1124,121,915.50,High



✅ Overstock risk analysis completed successfully.


# Cell 7 — Smart Reorder Recommendation Engine

In [7]:
forecast_output["Recommended_Stock_Level"] = (
    forecast_output["Lead_Time_Demand"] +
    forecast_output["Safety_Stock"]
)

forecast_output["Reorder_Quantity"] = (
    forecast_output["Recommended_Stock_Level"] -
    forecast_output["Available_Inventory"]
).clip(lower=0)

forecast_output["Reorder_Quantity"] = (
    forecast_output["Reorder_Quantity"].round(0)
)

forecast_output["Action"] = np.select(
    [
        forecast_output["Overstock_Risk"] == "High",
        forecast_output["Stockout_Risk"] == "High",
        forecast_output["Stockout_Risk"] == "Medium",
        forecast_output["Current_Stock"] <=
        forecast_output["Reorder_Point"]
    ],
    [
        "Do Not Reorder - Excess Stock",
        "Urgent Reorder",
        "Plan Reorder",
        "Reorder"
    ],
    default="No Action"
)

forecast_output["Priority"] = np.select(
    [
        forecast_output["Action"] == "Urgent Reorder",
        forecast_output["Action"] == "Plan Reorder",
        forecast_output["Action"] == "Reorder",
        forecast_output["Action"] ==
        "Do Not Reorder - Excess Stock"
    ],
    [
        "Critical",
        "High",
        "Medium",
        "High"
    ],
    default="Low"
)

print("=" * 80)
print("SMART REORDER RECOMMENDATION ENGINE")
print("=" * 80)

print("\nAction Distribution:")

action_summary = (
    forecast_output["Action"]
    .value_counts()
    .rename_axis("Action")
    .reset_index(name="SKU_Day_Count")
)

display(action_summary)

print("\nPriority Distribution:")

priority_summary = (
    forecast_output["Priority"]
    .value_counts()
    .rename_axis("Priority")
    .reset_index(name="SKU_Day_Count")
)

display(priority_summary)

print("\nTop Urgent Reorder Recommendations:")

urgent_reorders = (
    forecast_output[
        forecast_output["Action"] == "Urgent Reorder"
    ]
    [
        [
            "Date",
            "SKU",
            "Product_Name",
            "Category",
            "Predicted_Demand",
            "Current_Stock",
            "On_Order",
            "Lead_Time_Days",
            "Safety_Stock",
            "Lead_Time_Demand",
            "Reorder_Quantity",
            "Action",
            "Priority"
        ]
    ]
    .sort_values(
        "Reorder_Quantity",
        ascending=False
    )
    .head(10)
)

display(urgent_reorders)

print("\nTop Excess Stock Recommendations:")

excess_stock = (
    forecast_output[
        forecast_output["Action"] ==
        "Do Not Reorder - Excess Stock"
    ]
    [
        [
            "Date",
            "SKU",
            "Product_Name",
            "Category",
            "Predicted_Demand",
            "Current_Stock",
            "Safety_Stock",
            "Excess_Inventory",
            "Action",
            "Priority"
        ]
    ]
    .sort_values(
        "Excess_Inventory",
        ascending=False
    )
    .head(10)
)

display(excess_stock)

print("\n✅ Smart reorder recommendation engine completed successfully.")

SMART REORDER RECOMMENDATION ENGINE

Action Distribution:


,Action,SKU_Day_Count
0,No Action,2100
1,Do Not Reorder - Excess Stock,1994
2,Urgent Reorder,1768
3,Plan Reorder,1176
4,Reorder,202



Priority Distribution:


,Priority,SKU_Day_Count
0,High,3170
1,Low,2100
2,Critical,1768
3,Medium,202



Top Urgent Reorder Recommendations:


,Date,SKU,Product_Name,Category,Predicted_Demand,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Lead_Time_Demand,Reorder_Quantity,Action,Priority
1156,2025-09-01,SKU017,Product 017,Home Decor,38.52,95,26,12,108,462.24,449.0,Urgent Reorder,Critical
3056,2025-10-09,SKU017,Product 017,Home Decor,36.86,95,26,12,108,442.32,429.0,Urgent Reorder,Critical
806,2025-08-25,SKU017,Product 017,Home Decor,34.86,95,26,12,108,418.32,405.0,Urgent Reorder,Critical
3406,2025-10-16,SKU017,Product 017,Home Decor,33.43,95,26,12,108,401.16,388.0,Urgent Reorder,Critical
3070,2025-10-09,SKU031,Product 031,Furniture,37.88,93,40,13,28,492.44,387.0,Urgent Reorder,Critical
3356,2025-10-15,SKU017,Product 017,Home Decor,32.83,95,26,12,108,393.96,381.0,Urgent Reorder,Critical
3006,2025-10-08,SKU017,Product 017,Home Decor,32.77,95,26,12,108,393.24,380.0,Urgent Reorder,Critical
3420,2025-10-16,SKU031,Product 031,Furniture,35.60,93,40,13,28,462.80,358.0,Urgent Reorder,Critical
1206,2025-09-02,SKU017,Product 017,Home Decor,30.35,95,26,12,108,364.20,351.0,Urgent Reorder,Critical
1170,2025-09-01,SKU031,Product 031,Furniture,34.81,93,40,13,28,452.53,348.0,Urgent Reorder,Critical



Top Excess Stock Recommendations:


,Date,SKU,Product_Name,Category,Predicted_Demand,Current_Stock,Safety_Stock,Excess_Inventory,Action,Priority
7114,2025-12-29,SKU025,Product 025,Storage,5.15,1124,121,930.90,Do Not Reorder - Excess Stock,High
6764,2025-12-22,SKU025,Product 025,Storage,5.17,1124,121,930.62,Do Not Reorder - Excess Stock,High
7164,2025-12-30,SKU025,Product 025,Storage,5.39,1124,121,927.54,Do Not Reorder - Excess Stock,High
6664,2025-12-20,SKU025,Product 025,Storage,5.55,1124,121,925.30,Do Not Reorder - Excess Stock,High
6814,2025-12-23,SKU025,Product 025,Storage,5.57,1124,121,925.02,Do Not Reorder - Excess Stock,High
7014,2025-12-27,SKU025,Product 025,Storage,5.92,1124,121,920.12,Do Not Reorder - Excess Stock,High
6714,2025-12-21,SKU025,Product 025,Storage,6.04,1124,121,918.44,Do Not Reorder - Excess Stock,High
7214,2025-12-31,SKU025,Product 025,Storage,6.09,1124,121,917.74,Do Not Reorder - Excess Stock,High
6864,2025-12-24,SKU025,Product 025,Storage,6.16,1124,121,916.76,Do Not Reorder - Excess Stock,High
3914,2025-10-26,SKU025,Product 025,Storage,6.25,1124,121,915.50,Do Not Reorder - Excess Stock,High



✅ Smart reorder recommendation engine completed successfully.


# Cell 8 — Inventory Intelligence Score + Final Forecast Dataset

In [8]:
forecast_output["Inventory_Health_Score"] = 100

forecast_output.loc[
    forecast_output["Stockout_Risk"] == "Medium",
    "Inventory_Health_Score"
] -= 20

forecast_output.loc[
    forecast_output["Stockout_Risk"] == "High",
    "Inventory_Health_Score"
] -= 40

forecast_output.loc[
    forecast_output["Overstock_Risk"] == "Medium",
    "Inventory_Health_Score"
] -= 15

forecast_output.loc[
    forecast_output["Overstock_Risk"] == "High",
    "Inventory_Health_Score"
] -= 30

forecast_output.loc[
    forecast_output["Action"] == "Urgent Reorder",
    "Inventory_Health_Score"
] -= 10

forecast_output["Inventory_Health_Score"] = (
    forecast_output["Inventory_Health_Score"]
    .clip(0, 100)
)

forecast_output["Health_Level"] = np.select(
    [
        forecast_output["Inventory_Health_Score"] >= 80,
        forecast_output["Inventory_Health_Score"] >= 60,
        forecast_output["Inventory_Health_Score"] >= 40
    ],
    [
        "Healthy",
        "Watch",
        "At Risk"
    ],
    default="Critical"
)

final_forecast_columns = [
    "Date",
    "SKU",
    "Product_Name",
    "Category",
    "Subcategory",
    "Predicted_Demand",
    "Current_Stock",
    "On_Order",
    "Lead_Time_Days",
    "Safety_Stock",
    "Reorder_Point",
    "Lead_Time_Demand",
    "Available_Inventory",
    "Days_of_Inventory",
    "Stock_Coverage_After_Order",
    "Stockout_Gap",
    "Stockout_Risk",
    "Excess_Inventory",
    "Overstock_Risk",
    "Recommended_Stock_Level",
    "Reorder_Quantity",
    "Stock_Status",
    "Action",
    "Priority",
    "Inventory_Health_Score",
    "Health_Level"
]

final_forecast_df = forecast_output[
    final_forecast_columns
].copy()

print("=" * 80)
print("INVENTORY INTELLIGENCE SCORE")
print("=" * 80)

print("\nHealth Level Distribution:")

health_level_summary = (
    final_forecast_df["Health_Level"]
    .value_counts()
    .rename_axis("Health_Level")
    .reset_index(name="SKU_Day_Count")
)

display(health_level_summary)

print("\nAverage Inventory Health Score:")

print(
    round(
        final_forecast_df["Inventory_Health_Score"].mean(),
        2
    )
)

print("\nFinal Forecast Dataset Shape:")

print(
    final_forecast_df.shape
)

print("\nFinal Dataset Sample:")

display(
    final_forecast_df.head(10)
)

print("\nMissing Values:")

missing_values = (
    final_forecast_df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_values[
        missing_values > 0
    ].to_frame("Missing_Count")
)

print("\n✅ Inventory intelligence dataset created successfully.")

INVENTORY INTELLIGENCE SCORE

Health Level Distribution:


,Health_Level,SKU_Day_Count
0,Healthy,3478
1,Watch,1994
2,At Risk,1768



Average Inventory Health Score:
74.35

Final Forecast Dataset Shape:
(7240, 26)

Final Dataset Sample:


,Date,SKU,Product_Name,Category,Subcategory,Predicted_Demand,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,...,Stockout_Risk,Excess_Inventory,Overstock_Risk,Recommended_Stock_Level,Reorder_Quantity,Stock_Status,Action,Priority,Inventory_Health_Score,Health_Level
0,2025-08-09,SKU011,Product 011,Furniture,Chair,5.13,753,269,3,179,...,Low,558.61,High,194.39,0.0,Sufficient Stock,Do Not Reorder - Excess Stock,High,70,Watch
1,2025-08-09,SKU012,Product 012,Home Decor,Table,11.96,47,60,12,16,...,High,0.00,Low,159.52,53.0,Reorder Required,Urgent Reorder,Critical,50,At Risk
2,2025-08-09,SKU013,Product 013,Kitchen,Cushion,11.63,151,95,4,32,...,Low,72.48,Low,78.52,0.0,Sufficient Stock,No Action,Low,100,Healthy
3,2025-08-09,SKU014,Product 014,Lighting,Cookware,11.63,309,334,10,130,...,Low,62.70,Low,246.30,0.0,Sufficient Stock,No Action,Low,100,Healthy
4,2025-08-09,SKU015,Product 015,Storage,Lamp,11.27,187,245,9,119,...,Medium,0.00,Low,220.43,0.0,Reorder Required,Plan Reorder,High,80,Healthy
5,2025-08-09,SKU016,Product 016,Furniture,Shelf,12.42,255,187,11,30,...,Low,88.38,Medium,166.62,0.0,Sufficient Stock,No Action,Low,85,Healthy
6,2025-08-09,SKU017,Product 017,Home Decor,Cabinet,11.60,95,26,12,108,...,High,0.00,Low,247.20,126.0,Reorder Required,Urgent Reorder,Critical,50,At Risk
7,2025-08-09,SKU018,Product 018,Kitchen,Rug,10.46,235,102,8,66,...,Low,85.32,Medium,149.68,0.0,Sufficient Stock,No Action,Low,85,Healthy
8,2025-08-09,SKU019,Product 019,Lighting,Organizer,11.13,227,114,14,25,...,Low,46.18,Low,180.82,0.0,Sufficient Stock,No Action,Low,100,Healthy
9,2025-08-09,SKU020,Product 020,Storage,Sofa,10.18,1002,353,3,129,...,Low,842.46,High,159.54,0.0,Sufficient Stock,Do Not Reorder - Excess Stock,High,70,Watch



Missing Values:


,Missing_Count



✅ Inventory intelligence dataset created successfully.


# Cell 9 — Inventory Engine Edge-Case Validation

In [9]:
print("=" * 80)
print("INVENTORY ENGINE EDGE-CASE VALIDATION")
print("=" * 80)

required_columns = [
    "Predicted_Demand",
    "Current_Stock",
    "On_Order",
    "Lead_Time_Days",
    "Safety_Stock",
    "Reorder_Point",
    "Lead_Time_Demand",
    "Available_Inventory",
    "Days_of_Inventory",
    "Stockout_Risk",
    "Excess_Inventory",
    "Overstock_Risk",
    "Reorder_Quantity",
    "Action",
    "Priority",
    "Inventory_Health_Score"
]

missing_required = [
    col for col in required_columns
    if col not in final_forecast_df.columns
]

print("\nMissing Required Columns:")

if missing_required:
    print(missing_required)
else:
    print("None")

print("\nMissing Values:")

missing_check = (
    final_forecast_df[required_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_check[
        missing_check > 0
    ].to_frame("Missing_Count")
)

print("\nZero Demand Records:")

zero_demand = (
    final_forecast_df["Predicted_Demand"] <= 0
).sum()

print(zero_demand)

print("\nZero Stock Records:")

zero_stock = (
    final_forecast_df["Current_Stock"] <= 0
).sum()

print(zero_stock)

print("\nNegative Inventory Values:")

inventory_columns = [
    "Current_Stock",
    "On_Order",
    "Safety_Stock",
    "Reorder_Point",
    "Reorder_Quantity",
    "Excess_Inventory"
]

negative_values = (
    final_forecast_df[inventory_columns] < 0
).sum()

display(
    negative_values[
        negative_values > 0
    ].to_frame("Negative_Count")
)

print("\nHigh Lead-Time Records:")

high_lead_time = (
    final_forecast_df["Lead_Time_Days"] >= 14
).sum()

print(high_lead_time)

print("\nCritical Priority Records:")

critical_records = (
    final_forecast_df["Priority"] == "Critical"
).sum()

print(critical_records)

print("\nReorder Quantity Validation:")

invalid_reorder = (
    final_forecast_df["Reorder_Quantity"] < 0
).sum()

print(
    "Negative Reorder Quantity:",
    invalid_reorder
)

print("\nHealth Score Validation:")

invalid_scores = (
    (final_forecast_df["Inventory_Health_Score"] < 0) |
    (final_forecast_df["Inventory_Health_Score"] > 100)
).sum()

print(
    "Invalid Health Scores:",
    invalid_scores
)

print("\nRisk Categories:")

print(
    "Stockout:",
    final_forecast_df["Stockout_Risk"]
    .dropna()
    .unique()
)

print(
    "Overstock:",
    final_forecast_df["Overstock_Risk"]
    .dropna()
    .unique()
)

print("\nActions:")

print(
    final_forecast_df["Action"]
    .dropna()
    .unique()
)

print("\n✅ Edge-case validation completed.")

INVENTORY ENGINE EDGE-CASE VALIDATION

Missing Required Columns:
None

Missing Values:


,Missing_Count



Zero Demand Records:
0

Zero Stock Records:
0

Negative Inventory Values:


,Negative_Count



High Lead-Time Records:
580

Critical Priority Records:
1768

Reorder Quantity Validation:
Negative Reorder Quantity: 0

Health Score Validation:
Invalid Health Scores: 0

Risk Categories:
Stockout: <ArrowStringArray>
['Low', 'High', 'Medium']
Length: 3, dtype: str
Overstock: <ArrowStringArray>
['High', 'Low', 'Medium']
Length: 3, dtype: str

Actions:
<ArrowStringArray>
['Do Not Reorder - Excess Stock',                'Urgent Reorder',
                     'No Action',                  'Plan Reorder',
                       'Reorder']
Length: 5, dtype: str

✅ Edge-case validation completed.


# Cell 10 — Save Final Inventory Intelligence Dataset

In [10]:
OUTPUT_DIR = PROJECT_ROOT / "models" / "forecast_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

forecast_output_path = (
    OUTPUT_DIR / "foresight_inventory_intelligence.csv"
)

final_forecast_df.to_csv(
    forecast_output_path,
    index=False
)

print("=" * 80)
print("FINAL INVENTORY INTELLIGENCE DATASET")
print("=" * 80)

print("\nOutput Path:")
print(forecast_output_path)

print("\nDataset Shape:")
print(final_forecast_df.shape)

print("\nUnique SKUs:")
print(final_forecast_df["SKU"].nunique())

print("\nDate Range:")
print(
    final_forecast_df["Date"].min(),
    "to",
    final_forecast_df["Date"].max()
)

print("\nColumns:")
print(len(final_forecast_df.columns))

print("\nFile Size:")
print(
    round(
        forecast_output_path.stat().st_size / (1024 * 1024),
        2
    ),
    "MB"
)

print("\n✅ Final inventory intelligence dataset saved successfully.")

FINAL INVENTORY INTELLIGENCE DATASET

Output Path:
d:\Zidio project\FORESIGHT Project\models\forecast_output\foresight_inventory_intelligence.csv

Dataset Shape:
(7240, 26)

Unique SKUs:
50

Date Range:
2025-08-09 00:00:00 to 2025-12-31 00:00:00

Columns:
26

File Size:
1.47 MB

✅ Final inventory intelligence dataset saved successfully.
